In [1]:
# Titanic Dataset Features:
# survived: Whether the passenger survived (0 = No, 1 = Yes).
# pclass: Ticket class (1 = 1st, 2 = 2nd, 3 = 3rd).
# sex: Gender of the passenger.
# age: Age of the passenger in years. Some values are missing.
# sibsp: Number of siblings/spouses aboard the Titanic.
# parch: Number of parents/children aboard the Titanic.
# fare: Passenger fare.
# embarked: Port of embarkation (C = Cherbourg; Q = Queenstown; S = Southampton).
# class: Duplicate of 'pclass' (used for plotting by Seaborn).
# who: Describes whether the passenger is a man, woman, or child.
# adult_male: Indicates whether the passenger is an adult male (True/False).
# deck: The deck the passenger was on (missing for many passengers).
# embark_town: The name of the town where the passenger boarded.
# alive: Indicator of whether the passenger survived (Yes/No, derived from 'survived').
# alone: Indicates whether the passenger was traveling alone (True/False).

In [108]:
import pandas as pd
import sklearn
from sklearn import model_selection
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

In [85]:
# load the dataset

titanic_data ="data/titanic.csv"
titanic_df = pd.read_csv(titanic_data)
titanic_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [65]:
titanic_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [66]:
titanic_df.select_dtypes(include = 'object').columns
# titanic_df.select_dtypes(include = 'object').columns.tolist()
# titanic_df.select_dtypes(include = ['object', 'int64'])
# titanic_df.select_dtypes(include = ['object', 'int64'])[:5]
# titanic_df.select_dtypes(exclude = ['object'])[:5]

/var/folders/19/_m__1vl122lfw6k451rx98840000gp/T/ipykernel_89611/2323604067.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  titanic_df.select_dtypes(include = 'object').columns


Index(['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked'], dtype='str')

In [97]:
titanic_df['Embarked'].value_counts()
# titanic_df['Embarked'].isna().sum()
# titanic_df['Age'].isna().sum()
# titanic_df['Cabin'].isna().sum()

Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

In [98]:
#Pre Processing

titanic_df.Embarked = titanic_df.Embarked.fillna(titanic_df['Embarked'].mode()[0])

#mean : average value of the data
#median : middle value of the data
#mode : most frequently occurring value in the data

In [86]:
median_age = titanic_df.Age.median()
median_fare = titanic_df.Fare.median()
print(median_age)
print(median_fare)

titanic_df['Age']=titanic_df['Age'].fillna(median_age)
titanic_df.Fare.fillna(median_fare, inplace = True)

28.0
14.4542


/var/folders/19/_m__1vl122lfw6k451rx98840000gp/T/ipykernel_89611/3616017433.py:7: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  titanic_df.Fare.fillna(median_fare, inplace = True)


0       7.2500
1      71.2833
2       7.9250
3      53.1000
4       8.0500
        ...   
886    13.0000
887    30.0000
888    23.4500
889    30.0000
890     7.7500
Name: Fare, Length: 891, dtype: float64

In [88]:
titanic_df.drop('Cabin', axis = 1, inplace = True)

In [90]:
titanic_df['FamilySize'] = titanic_df['SibSp'] + titanic_df['Parch'] + 1
titanic_df['GenderClass'] = titanic_df.apply(lambda x: 'child' if x['Age'] < 15 else x['Sex'],axis = 1) 

In [91]:
# titanic_df[titanicdf.Age<15].head(2)
titanic_df['GenderClass'].value_counts()
# titanic_df['Embarked'].value_counts()

GenderClass
male      538
female    275
child      78
Name: count, dtype: int64

In [92]:
# titanic_df["Pclass"].unique()
titanic_df.Pclass.unique()
#titanic_df['Embarked'].value_counts()


array([3, 1, 2])

In [93]:
titanic_df.groupby("Pclass").count()

,PassengerId,Survived,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,FamilySize,GenderClass
Pclass,,,,,,,,,,,,
1,216,216,216,216,216,216,216,216,216,214,216,216
2,184,184,184,184,184,184,184,184,184,184,184,184
3,491,491,491,491,491,491,491,491,491,491,491,491


In [75]:
#dict(titanic_df.groupby("Pclass")['PassengerId'].count())

In [76]:
#titanic_df.groupby('Embarked').count()

In [94]:
titanic_df.groupby('Survived').count()
#titanic_df.groupby('Survived')['PassengerId'].count()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,FamilySize,GenderClass
Survived,,,,,,,,,,,,
0,549,549,549,549,549,549,549,549,549,549,549,549
1,342,342,342,342,342,342,342,342,342,340,342,342


In [95]:
#frequency of unique values
titanic_df['Survived'].value_counts()
#titanic_df['Embarked'].value_counts()

Survived
0    549
1    342
Name: count, dtype: int64

In [99]:
#checking for missing values
print("Missing values in each column:")
print(titanic_df.isnull().sum())

Missing values in each column:
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
FamilySize     0
GenderClass    0
dtype: int64


In [100]:
titanic_df.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,FamilySize,GenderClass
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S,2,male
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C,2,female
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S,1,female


In [102]:
titanic_df=pd.get_dummies(titanic_df, columns=['GenderClass', 'Embarked'], drop_first=True)
titanic_df.head(3)

KeyError: "None of [Index(['GenderClass', 'Embarked'], dtype='str')] are in the [columns]"

In [103]:
titanic_df1 = titanic_df.drop(['Name','Sex','Ticket'],axis=1)

In [104]:
X = titanic_df1.drop('Survived', axis=1)
y = titanic_df1['Survived']

In [105]:
y.head()

0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64

In [109]:
X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=0.2, random_state=42)

In [110]:
model = LogisticRegression(solver='liblinear')
model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [111]:
predictions = model.predict(X_test)

In [112]:
confusion_matrix(y_test, model.predict(X_test))

array([[90, 15],
       [23, 51]])

In [113]:
report = classification_report(y_test, predictions)
print(report)

              precision    recall  f1-score   support

           0       0.80      0.86      0.83       105
           1       0.77      0.69      0.73        74

    accuracy                           0.79       179
   macro avg       0.78      0.77      0.78       179
weighted avg       0.79      0.79      0.79       179



classification_report gives a class-wise performance summary
It reports 4 key metrics per class:

Precision
Recall
F1-score
Support
Precision (per class) : Of all passengers predicted as a given class, how many were correct?

For class1 (Survived) : precision = 0.76

Out of all passengers predicted as survived, 76% actually survived
Recall (per class) : Of all actual passengers in a class, how many did the model correctly identify? For class1 (Survived) : recall = 0.64

Out of all passengers who actually survived, the model identified 64%
36% of survivors were missed (false negatives)
F1-score : Harmonic mean of Precision and Recall F1 = 2 × (Precision × Recall) / (Precision + Recall)

For class1 (Survived) : f1-score = 0.69

Balanced measure when both precision and recall matter
Useful when classes are imbalanced (Titanic is slightly imbalanced)
Support : support = 74

There are 74 actual survivors in the test set
Metrics for this class are calculated using these 74 rows
Accuracy (overall) : accuracy = 0.79

Model correctly predicted 79% of all passengers Accuracy alone is misleading if:
Dataset is imbalanced
One class dominates (non‑survivors in Titanic)
Macro vs Weighted Average Macro Avg = simple average across classes

Treats survivors and non‑survivors equally
Ignores class imbalance
Weighted Avg = weighted by support

Accounts for class imbalance

In [115]:
model.coef_
# Positive coefficient → increases survival probability
# Negative coefficient → decreases survival probability
# Example interpretation:
# Sex_female = +2.1 → Being female strongly increases survival chance
# Pclass_3 = -1.3 → 3rd class reduces survival probability

array([[ 4.55101029e-04, -7.29385102e-01, -1.02827495e-02,
        -9.42876424e-01, -7.77732177e-01,  5.01537814e-03,
         5.60638492e-01,  5.75966074e-01, -2.49678202e+00,
        -2.39517367e-01, -3.23973397e-01]])

In [116]:
X_test.head()

,PassengerId,Pclass,Age,SibSp,Parch,Fare,FamilySize,GenderClass_female,GenderClass_male,Embarked_Q,Embarked_S
709,710,3,28.0,1,1,15.2458,3,False,True,False,False
439,440,2,31.0,0,0,10.5000,1,False,True,False,True
840,841,3,20.0,0,0,7.9250,1,False,True,False,True
720,721,2,6.0,0,1,33.0000,2,False,False,False,True
39,40,3,14.0,1,0,11.2417,2,False,False,False,False


In [117]:
y_test.head()

709    1
439    0
840    0
720    1
39     1
Name: Survived, dtype: int64